# MovieMate: IMDb Conversational Chatbot

This notebook documents the dataset exploration, preprocessing, retrieval design, and conversational prototype built from the provided IMDb Top 1000 dataset.

## 1. Load the dataset

The project uses a cleaned version of the provided CSV and imports the local project modules.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'movie_mate').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_mate import MovieChatbot, compute_insights, load_movies

movies = load_movies(project_root / 'data' / 'imdb_top_1000.csv')
len(movies)

## 2. Dataset exploration

We summarize the dataset to understand ratings, runtimes, common genres, directors, and missing values.

In [ ]:
insights = compute_insights(movies)
insights['summary']

In [ ]:
insights['top_genres'][:5], insights['top_directors'][:5], insights['missing_fields']

## 3. Preprocessing notes

- Standardized text fields for search.
- Parsed runtime, votes, gross, and metascore into numeric values where possible.
- Repaired the malformed `Apollo 13` release year.
- Preserved missing metadata safely.


## 4. Retrieval and conversational logic

The chatbot combines structured query parsing with TF-IDF retrieval over title, genre, cast, director, and overview text. When configured with `OPENAI_API_KEY`, it can also use the OpenAI Responses API for natural-language replies and OpenAI embeddings for semantic retrieval. Follow-up messages can refine the prior query state, and the app stores a lightweight user preference memory profile.

In [ ]:
bot = MovieChatbot(movies)
session = None
for query in [
    'Suggest sci-fi movies released after 2010',
    'Only those with IMDb rating above 8.0',
    'Movies similar to Inception',
    'Who directed The Godfather?'
]:
    response = bot.respond(query, session)
    session = response['session_id']
    print(query)
    print(response['reply'])
    print([card['title'] for card in response['cards'][:3]])
    print('---')

## 5. Evaluation and reflection

Strengths:
- Handles natural-language filters for genre, year, actor, director, rating, and runtime.
- Supports multi-turn refinement and title detail lookups.
- Supports optional OpenAI-based natural-language generation, semantic retrieval, and stored preference memory.
- Stays lightweight and reproducible without external dependencies in fallback mode.

Limitations:
- OpenAI-backed features require API keys and optional embedding-cache preparation.
- The deployed free-tier environment uses ephemeral storage, so memory and embedding cache persistence are prototype-grade.
- The dataset is still limited to the IMDb Top 1000 entries supplied in the CSV unless an external API ingestion workflow is used.
